In [2]:
pip install pandas sqlalchemy pymysql scipy statsmodels

Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
from sqlalchemy import create_engine, URL

connection_url = URL.create(
    "mysql+pymysql",
    username="root",
    password="root",
    host="localhost",
    port=3306,
    database="abtessting"
)

engine = create_engine(connection_url)

print("MySQL connection configured successfully!")

MySQL connection configured successfully!


In [4]:
query = "SELECT * FROM users"

df = pd.read_sql(query, con=engine)

print("Data imported successfully!")

Data imported successfully!


In [5]:
df.head()

,user_id,experiment_group,experiment_date,device_type,traffic_source,country,converted,revenue,checkout_time_sec
0,200001,B,2026-09-15,Desktop,Organic,India,0,0.00,
1,200002,B,2026-09-05,Desktop,Social,United States,1,869.98,144.1
2,200003,A,2026-09-09,Mobile,Email,United Kingdom,0,0.00,
3,200004,B,2026-09-23,Desktop,Organic,India,0,0.00,
4,200005,B,2026-09-10,Desktop,Paid Ads,United States,0,0.00,


In [6]:
df.shape

(30000, 9)

In [7]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   user_id            30000 non-null  int64  
 1   experiment_group   30000 non-null  object 
 2   experiment_date    30000 non-null  object 
 3   device_type        30000 non-null  object 
 4   traffic_source     30000 non-null  object 
 5   country            30000 non-null  object 
 6   converted          30000 non-null  int64  
 7   revenue            30000 non-null  float64
 8   checkout_time_sec  30000 non-null  object 
dtypes: float64(1), int64(2), object(6)
memory usage: 2.1+ MB


In [8]:
df.isnull().sum()

user_id              0
experiment_group     0
experiment_date      0
device_type          0
traffic_source       0
country              0
converted            0
revenue              0
checkout_time_sec    0
dtype: int64

In [9]:
df["experiment_group"].value_counts()

experiment_group
B    15000
A    15000
Name: count, dtype: int64

In [10]:
print("Total users:", len(df))
print("Total columns:", len(df.columns))
print("Group A users:", (df["experiment_group"] == "A").sum())
print("Group B users:", (df["experiment_group"] == "B").sum())

Total users: 30000
Total columns: 9
Group A users: 15000
Group B users: 15000


#  1: Calculate conversion rates in Pandas

In [12]:
conversion_summary = df.groupby("experiment_group").agg(
    total_users=("user_id", "count"),
    total_purchases=("converted", "sum")
).reset_index()

conversion_summary["conversion_rate"] = (
    conversion_summary["total_purchases"]
    / conversion_summary["total_users"]
) * 100

conversion_summary

,experiment_group,total_users,total_purchases,conversion_rate
0,A,15000,755,5.033333
1,B,15000,846,5.640000


# 2: Define the hypotheses
Before testing, we need to define what we want to prove.
- Null hypothesis (H₀): Group A and Group B have the same purchase conversion rate.
- Alternative hypothesis (H₁): Group A and Group B have different purchase conversion rates.

## We’ll use a 5% significance level (\(\alpha = 0.05\)).

# 3: Perform the two-proportion z-test

In [14]:
from statsmodels.stats.proportion import proportions_ztest

# Total purchases in each group
purchases = [
    df.loc[df["experiment_group"] == "A", "converted"].sum(),
    df.loc[df["experiment_group"] == "B", "converted"].sum()
]

# Total users in each group
users = [
    (df["experiment_group"] == "A").sum(),
    (df["experiment_group"] == "B").sum()
]

# Perform the two-proportion z-test
z_stat, p_value = proportions_ztest(
    count=purchases,
    nobs=users,
    alternative="two-sided"
)

print("Z-statistic:", round(z_stat, 4))
print("P-value:", round(p_value, 4))

if p_value < 0.05:
    print("Reject the null hypothesis.")
    print("The conversion rates are statistically significantly different.")
else:
    print("Fail to reject the null hypothesis.")
    print("There is insufficient evidence of a significant difference.")

Z-statistic: -2.3375
P-value: 0.0194
Reject the null hypothesis.
The conversion rates are statistically significantly different.


# RESULT
- P-value = 0.0194: This is lower than 0.05, so we reject the null hypothesis.
- Z-statistic = -2.3375: The negative sign indicates that Group A's conversion rate is lower than Group B's, given the order of groups in your test.
- Conclusion: There is statistically significant evidence that the conversion rates of the two checkout pages differ.

Your results suggest that the redesigned checkout page (Group B) performs better than the existing page (Group A) in terms of purchase conversion. Before recommending a full rollout, we should also check the actual conversion rates and the size of the improvement.

# 4 Calculate conversion uplift

In [15]:
rate_A = (
    df.loc[df["experiment_group"] == "A", "converted"].mean()
) * 100

rate_B = (
    df.loc[df["experiment_group"] == "B", "converted"].mean()
) * 100

absolute_uplift = rate_B - rate_A

relative_uplift = (
    (rate_B - rate_A) / rate_A
) * 100

print(f"Group A Conversion Rate: {rate_A:.2f}%")
print(f"Group B Conversion Rate: {rate_B:.2f}%")
print(f"Absolute Uplift: {absolute_uplift:.2f} percentage points")
print(f"Relative Uplift: {relative_uplift:.2f}%")

Group A Conversion Rate: 5.03%
Group B Conversion Rate: 5.64%
Absolute Uplift: 0.61 percentage points
Relative Uplift: 12.05%


# Result
- Group A Conversion Rate: 5.03%
- Group B Conversion Rate: 5.64%
- Absolute Uplift: 0.61 percentage points
- Relative Uplift: 12.05%
Conclusion: Group B's redesigned checkout page achieved a 12.05% relative improvement in conversion rate compared with Group A's existing page. Your earlier p-value of 0.0194 indicates that this difference is statistically significant at the 5% level.

 # 95% confidence interval for the difference in conversion rates.

In [16]:
from statsmodels.stats.proportion import confint_proportions_2indep

# Count purchases in each group
purchases_A = df.loc[
    df["experiment_group"] == "A", "converted"
].sum()

purchases_B = df.loc[
    df["experiment_group"] == "B", "converted"
].sum()

# Count users in each group
users_A = (df["experiment_group"] == "A").sum()
users_B = (df["experiment_group"] == "B").sum()

# Calculate 95% confidence interval
ci_low, ci_high = confint_proportions_2indep(
    count1=purchases_B,
    nobs1=users_B,
    count2=purchases_A,
    nobs2=users_A,
    method="newcomb",
    compare="diff",
    alpha=0.05
)

print(f"95% Confidence Interval: "
      f"({ci_low * 100:.2f}, {ci_high * 100:.2f}) percentage points")

95% Confidence Interval: (0.10, 1.12) percentage points


# Business Conclusion – A/B Testing

The A/B testing results show that the redesigned checkout page (Group B) performed better than the existing page (Group A). Group B achieved a **5.64% conversion rate**, compared with **5.03%** for Group A, representing a **12.05% relative improvement**. The difference was statistically significant, with a p-value of **0.0194** and a 95% confidence interval of **0.10 to 1.12 percentage points** for the conversion-rate improvement. Additionally, revenue per user increased from **101.65 to 115.36**, a **13.49% increase**, with a statistically significant p-value of **0.0251**. Therefore, Group B is the preferred option based on the observed conversion and revenue results, subject to validating the experiment quality and business impact before a full rollout.

## In simple words, The redesigned checkout page converts more users into customers and generates more revenue per user. Both improvements are statistically significant in this simulated dataset.